**CI twin of `ch04-forward-pass.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.25,
    random_state=42, stratify=digits.target)
net = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                    random_state=0, max_iter=2000).fit(Xtr, ytr)

W1, W2 = net.coefs_          # note: sklearn stores these as (n_in, n_out)
b1, b2 = net.intercepts_

x = Xte[0]                               # truth: a 1
hidden = np.maximum(0, x @ W1 + b1)      # the 16 clerks report
logits = hidden @ W2 + b2                # 10 judges score
print(f"x: {x.shape}   hidden: {hidden.shape}   logits: {logits.shape}")
print(f"logits: {np.round(logits, 1)}")
print(f"our verdict: {int(np.argmax(logits))}   "
      f"sklearn's: {int(net.predict(Xte[[0]])[0])}   truth: {yte[0]}")

In [ ]:
def hand_forward(X):
    H = np.maximum(0, X @ W1 + b1)
    return H @ W2 + b2

our_verdicts = np.argmax(hand_forward(Xte), axis=1)
agreement = int((our_verdicts == net.predict(Xte)).sum())
print(f"agreement with sklearn: {agreement} / {len(Xte)}")
print(f"our hand-made accuracy: {(our_verdicts == yte).mean():.3f}")

In [ ]:
import time

t0 = time.perf_counter()
loop_preds = [np.argmax(np.maximum(0, xi @ W1 + b1) @ W2 + b2)
              for xi in Xte]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
batch_preds = np.argmax(hand_forward(Xte), axis=1)
t_batch = time.perf_counter() - t0

print(f"python loop: {t_loop * 1000:6.1f} ms")
print(f"one batch:   {t_batch * 1000:6.2f} ms   "
      f"(~{t_loop / t_batch:.0f}x faster)")

In [ ]:
try:
    Xte @ W2        # (450×64) @ (16×10): 64 ≠ 16 — illegal on purpose
except ValueError as e:
    print(f"ValueError: {str(e)[:70]}…")

In [ ]:
def forward(X, params):
    """Batched forward pass. params = [(W1, b1), (W2, b2), ...]."""
    out = X
    for i, (W, b) in enumerate(params):
        out = out @ W + b
        if i < len(params) - 1:          # bend between layers,
            out = np.maximum(0, out)     # never after the last
    return out

logits_all = forward(Xte, [(W1, b1), (W2, b2)])
print(f"agreement, again: "
      f"{int((np.argmax(logits_all, axis=1) == net.predict(Xte)).sum())} / 450")

In [ ]:
hidden = np.maximum(0, x @ W1 + b1)
logits = hidden @ W2 + b2

run_tests([
    ("sixteen clerk reports", hidden.shape, (16,)),
    ("ten judges' scores", logits.shape, (10,)),
    ("the verdict", int(np.argmax(logits)), 1),
    ("matches the appliance", int(np.argmax(logits)),
     int(net.predict(Xte[[0]])[0])),
])

In [ ]:
def forward(X, params):
    out = X
    for i, (W, b) in enumerate(params):
        out = out @ W + b
        if i < len(params) - 1:
            out = np.maximum(0, out)
    return out

Wa = np.array([[1.0, -1.0], [0.0, 1.0]])
ba = np.array([0.0, 0.5])
Wb = np.array([[2.0], [1.0]])
bb = np.array([-1.0])
tiny = forward(np.array([[1.0, 2.0], [0.0, 0.0]]), [(Wa, ba), (Wb, bb)])

sk_params = list(zip(net.coefs_, net.intercepts_))
agreement = int((np.argmax(forward(Xte, sk_params), axis=1)
                 == net.predict(Xte)).sum())

run_tests([
    ("hand-checkable batch of two",
     [round(float(v), 1) for v in tiny.ravel()], [2.5, -0.5]),
    ("the full audit", agreement, 450),
])